# Projeto Completo (CRISP-DM) — Violência Doméstica no Distrito Federal (2024–2025)

**Pergunta de negócio:** *Quais Regiões Administrativas (RAs) do Distrito Federal concentram a maior parte dos registros de violência doméstica em 2025, e como esse cenário evoluiu em relação a 2024?*

**Metodologia:** CRISP-DM (*Cross-Industry Standard Process for Data Mining*)  
**Fonte oficial:** Secretaria de Segurança Pública do Distrito Federal (SSP-DF) — Relatórios RAC-001/2026, RAC-020/2026 e série histórica FSP (2018–2025).

| Fase CRISP-DM | Entregável neste Notebook |
|---|---|
| **1. Entendimento do Negócio** | Definição do problema, escopo, métricas e critérios de sucesso |
| **2. Entendimento dos Dados** | Inventário dos relatórios oficiais da SSP-DF e diagnóstico de qualidade |
| **3. Preparação dos Dados** | Pipeline de extração (texto/OCR), limpeza, padronização das 33 RAs e variáveis derivadas |
| **4. Modelagem** | Ranking 2025, Curva de Pareto, Variação 2024→2025, Correlação de Spearman e Clusterização K-Means |
| **5. Avaliação** | Auditoria de integridade (0% divergência), consistência sazonal e teste de robustez ($\rho = 0,992$) |
| **6. Implantação** | Conclusões executivas, limitações metodológicas e roadmap (normalização per capita) |

## Fase 1 & 2 — Entendimento do Negócio e dos Dados

### 1.1 Objetivos e Critérios de Sucesso
- **Objetivo Analítico:** Identificar o padrão de concentração espacial e a evolução interanual (2024 vs. 2025) dos registros policiais enquadrados na Lei Maria da Penha (Lei nº 11.340/2006) nas 33 Regiões Administrativas do DF.
- **Critérios de Sucesso Técnico:**
  1. **Integridade de Extração:** 0% de divergência entre a soma das RAs extraídas e os totais oficiais publicados pela SSP-DF.
  2. **Robustez Estrutural:** Correlação de postos de Spearman ($\rho$) $\ge 0,90$ entre os rankings de 2024 e 2025.
  3. **Rigor Metodológico:** Não confundir volume absoluto de ocorrências (demanda operacional) com taxa de risco individual (incidência per capita).

### 1.2 Ambiente e Carregamento da Base

In [1]:
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAIZ = Path('..').resolve()
sys.path.insert(0, str(RAIZ / 'src'))

BRUTOS = RAIZ / 'dados' / 'brutos'
PROCESSADOS = RAIZ / 'dados' / 'processados'

print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

## Fase 3 — Preparação dos Dados (ETL & Engenharia de Atributos)

Os relatórios oficiais da SSP-DF são disponibilizados apenas em PDF (parte em texto vetorial, parte em tabelas rasterizadas exigindo OCR via Tesseract). O pipeline abaixo:
1. Executa a extração e auditoria automática (`extracao.main`) caso a base processada ainda não exista;
2. Padroniza a nomenclatura das 33 RAs atuais;
3. Constrói variáveis derivadas (participação %, ranking, frequência acumulada de Pareto e variação interanual).

In [2]:
csv_processado = PROCESSADOS / 'violencia_domestica_ra.csv'

if not csv_processado.exists():
    import extracao
    tabelas, longa, base, log, revisoes = extracao.main(BRUTOS, PROCESSADOS)
else:
    base = pd.read_csv(csv_processado)
    log = pd.read_csv(PROCESSADOS / 'log_validacao.csv')

# Engenharia de atributos
base['total_periodo'] = base.groupby(['periodo', 'ano'])['registros'].transform('sum')
base['participacao_pct'] = (100 * base['registros'] / base['total_periodo']).round(2)
base['ranking'] = base.groupby(['periodo', 'ano'])['registros'].rank(ascending=False, method='min').astype(int)
base['n_ras_no_periodo'] = base.groupby(['periodo', 'ano'])['ra'].transform('count')
base['comparavel_33_RAs'] = base['n_ras_no_periodo'] == 33

# Tabela pivô (formato largo por RA)
anual = base[base['periodo'] == 'Anual'].pivot(index='ra', columns='ano', values='registros')
anual.columns = [f'reg_{c}' for c in anual.columns]

semestral = base[base['periodo'] == '1º semestre'].pivot(index='ra', columns='ano', values='registros')
semestral.columns = [f'reg_1sem_{c}' for c in semestral.columns]

df_ra = anual.join(semestral, how='outer')
df_ra['var_abs_2024_2025'] = df_ra['reg_2025'] - df_ra['reg_2024']
df_ra['var_pct_2024_2025'] = (100 * df_ra['var_abs_2024_2025'] / df_ra['reg_2024']).round(1)
df_ra['part_2025_pct'] = (100 * df_ra['reg_2025'] / df_ra['reg_2025'].sum()).round(2)
df_ra['ranking_2024'] = df_ra['reg_2024'].rank(ascending=False, method='min').astype(int)
df_ra['ranking_2025'] = df_ra['reg_2025'].rank(ascending=False, method='min').astype(int)
df_ra['delta_ranking'] = df_ra['ranking_2024'] - df_ra['ranking_2025']  # positivo = subiu no ranking

df_ra = df_ra.sort_values('ranking_2025')
df_ra['acumulado_2025_pct'] = df_ra['part_2025_pct'].cumsum().round(2)

print(f'Total DF 2024: {int(df_ra["reg_2024"].sum()):,} | Total DF 2025: {int(df_ra["reg_2025"].sum()):,} '
      f'(+{100*(df_ra["reg_2025"].sum()/df_ra["reg_2024"].sum() - 1):.1f}%)')
df_ra[['reg_2024', 'reg_2025', 'var_abs_2024_2025', 'var_pct_2024_2025', 
       'part_2025_pct', 'acumulado_2025_pct', 'ranking_2024', 'ranking_2025']].head(15)

## Fase 4 — Modelagem Analítica

Aplicação das técnicas analíticas para responder à pergunta de negócio:
- **4.1** Concentração Geográfica e Curva de Pareto (2025)
- **4.2** Evolução Interanual (2024 → 2025) e Dinâmica de Ranking
- **4.3** Segmentação das RAs via Clusterização (K-Means)

In [3]:
# 4.1 Marcos da Curva de Pareto (2025)
n_50 = (df_ra['acumulado_2025_pct'] <= 50.0).sum() + 1
top10_pct = df_ra.head(10)['part_2025_pct'].sum()
n_80 = (df_ra['acumulado_2025_pct'] <= 80.0).sum() + 1

print(f'Pareto 2025 -> {n_50} RAs concentram ~50% dos casos | Top 10 RAs = {top10_pct:.1f}% | {n_80} RAs atingem ~80%')

# Visualização: Ranking Top 15 + Curva Acumulada de Pareto
fig, ax1 = plt.subplots(figsize=(11, 5.5))
top15 = df_ra.head(15).reset_index()

sns.barplot(data=top15, x='ra', y='reg_2025', color='#2b5c8f', ax=ax1)
ax1.set_ylabel('Registros em 2025', color='#2b5c8f', fontweight='bold')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=40)

ax2 = ax1.twinx()
ax2.plot(top15['ra'], top15['acumulado_2025_pct'], color='#d95f02', marker='o', linewidth=2, label='% Acumulado')
ax2.axhline(50, color='gray', linestyle='--', linewidth=0.9, label='50% dos registros')
ax2.axhline(80, color='darkred', linestyle=':', linewidth=0.9, label='80% dos registros (Pareto)')
ax2.set_ylabel('Participação Acumulada (%)', color='#d95f02', fontweight='bold')
ax2.set_ylim(0, 100)

plt.title('Top 15 RAs em Violência Doméstica (2025) e Curva de Pareto', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [4]:
# 4.2 Clusterização K-Means (Segmentação por Volume 2025 e Variação % 2024-2025)
features = df_ra[['reg_2025', 'var_pct_2024_2025']].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ra['cluster_id'] = kmeans.fit_predict(X_scaled)

# Ordenar rótulos dos clusters pelo volume médio de 2025
ordem_clusters = df_ra.groupby('cluster_id')['reg_2025'].mean().sort_values(ascending=False).index
mapa_nomes = {
    ordem_clusters[0]: 'Grupo 1 — Alta Incidência Absoluta (Polos Críticos)',
    ordem_clusters[1]: 'Grupo 2 — Incidência Intermediária',
    ordem_clusters[2]: 'Grupo 3 — Baixo Volume Absoluto'
}
df_ra['perfil_cluster'] = df_ra['cluster_id'].map(mapa_nomes)

resumo_clusters = df_ra.groupby('perfil_cluster').agg(
    qtd_RAs=('reg_2025', 'count'),
    soma_2025=('reg_2025', 'sum'),
    media_2025=('reg_2025', 'mean'),
    var_media_pct=('var_pct_2024_2025', 'mean')
).round(1)
resumo_clusters['part_total_df_pct'] = (100 * resumo_clusters['soma_2025'] / df_ra['reg_2025'].sum()).round(1)
resumo_clusters

## Fase 5 — Avaliação e Validação Técnica

Antes da implantação, o modelo descritivo passa por três auditorias formais:
1. **Integridade da Extração:** Verificação de erros no log de validação;
2. **Consistência Sazonal:** Proporção do 1º semestre de 2025 sobre o total anual de 2025 (faixa esperada: 40% a 60%);
3. **Robustez do Ranking (Spearman):** Estabilidade da ordenação entre 2024 e 2025.

In [5]:
# 1. Auditoria de log
n_erros = (log['tipo'] == 'ERRO').sum()

# 2. Consistência sazonal (1º sem 2025 / Anual 2025)
df_ra['peso_1sem_2025_pct'] = (100 * df_ra['reg_1sem_2025'] / df_ra['reg_2025']).round(1)
peso_df_1sem = 100 * df_ra['reg_1sem_2025'].sum() / df_ra['reg_2025'].sum()
fora_faixa = df_ra[(df_ra['peso_1sem_2025_pct'] < 40) | (df_ra['peso_1sem_2025_pct'] > 60)]

# 3. Correlação de postos de Spearman (2024 vs 2025)
rho, p_valor = spearmanr(df_ra['reg_2024'], df_ra['reg_2025'])

print('=== RELATÓRIO DE AVALIAÇÃO (FASE 5) ===')
print(f'1. Erros de extração vs. PDFs oficiais : {n_erros} divergências (APROVADO)')
print(f'2. Sazonalidade 1º Sem/2025 (DF)       : {peso_df_1sem:.1f}% | RAs fora de [40%-60%]: {len(fora_faixa)} ({list(fora_faixa.index)})')
print(f'3. Correlação de Spearman (2024-2025)  : rho = {rho:.4f} (p = {p_valor:.2e}) -> Meta >= 0.90 (APROVADO)')

assert n_erros == 0, 'Falha na auditoria de integridade'
assert rho >= 0.90, 'Falha no critério de estabilidade do ranking'

## Fase 6 — Implantação, Conclusões e Próximas Iterações

### Principais Conclusões
1. **Crescimento Geral:** Os registros de violência doméstica no DF aumentaram **6,2%** entre 2024 ($21.721$) e 2025 ($23.066$).
2. **Alta Concentração Espacial:** Apenas **7 RAs** concentram metade ($50\%$) de todos os casos do DF, e **15 RAs** reúnem **$80\%$** das ocorrências.
3. **Polos Críticos Persistentes:** **Ceilândia** ($3.135$ casos; $13,6\%$ do DF), **Planaltina** ($1.951$; $8,5\%$), **Samambaia** ($1.809$; $7,8\%$), **Taguatinga** ($1.339$; $5,8\%$) e **Recanto das Emas** ($1.303$; $5,6\%$) lideram o ranking. A composição do Top 10 manteve-se inalterada entre 2024 e 2025 ($\rho = 0,992$).

### Limitações e Roadmap (Iteração 2.0)
- **Volume Absoluto vs. Taxa Per Capita:** Os resultados refletem a demanda bruta por atendimento policial e delegacias especializadas (DEAMs). Regiões mais populosas tendem a registrar maiores volumes absolutos. Na próxima iteração do ciclo CRISP-DM, os dados serão cruzados com a **PDAD (IPEDF)** para calcular a **taxa por 100 mil mulheres**.